# 01. 데이터 준비 · 구조 설명 · EDA — Amateur Drawings Dataset

**담당:** 데이터·평가 (1주차)
**목표:** ① 데이터셋을 무료 Colab에서 다룰 수 있는 크기로 받아오고 ② 데이터 구조를 정리하고 ③ EDA로 특성을 파악한 뒤 ④ 실험용 서브셋(개발/평가)을 만들어 팀원이 바로 쓸 수 있게 저장한다.

| 단계 | 내용 | 산출물 |
|---|---|---|
| 0 | 설정 · Drive 연결 | – |
| 1 | 어노테이션 다운로드 (288MB) | `amateur_drawings_annotations.json` |
| 2 | 데이터 구조 설명 | (이 노트북 본문) |
| 3 | 메타데이터 EDA (전체 178,166장) | `figures/*.png` |
| 4 | 이미지 일부만 스트리밍 다운로드 (전체 48GB 중 앞부분 N장) | `pool/` |
| 5 | 공통 유틸 (`ad_utils.py`) — 마스크 변환, IoU, 경계 F1 | `ad_utils.py` |
| 6 | 이미지 수준 EDA — 조명·그림자·배경 복잡도 | `pool_features.csv` |
| 7 | 라벨 품질 점검 · 서브셋 선정 · 중복 제거 · dev/test 분할 | `subset/` |
| 8 | 평가용 마스크 수동 검토 | `subset/test_review.csv` |
| 9 | 기본 방법 sanity check (Otsu IoU) | 표 |

> ⚠️ **전체 이미지(48GB tar)는 받지 않는다.** tar를 스트리밍으로 읽다가 필요한 장수만큼 모이면 멈춘다. 첫 실행 기준 약 5~10분.

## 0. 설정

In [ ]:
import os, sys, json, time, random, shutil, tarfile, urllib.request, hashlib
from collections import Counter
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = True          # 팀 공유 Drive에 저장하려면 True (권장)

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/cv_project/data'   # ← 팀 공유 폴더 경로로 바꾸기
else:
    BASE_DIR = os.environ.get('AD_BASE', '/content/cv_project/data')

# 대량의 작은 파일은 Drive에 쓰면 느리므로, 후보 이미지 풀은 로컬 디스크에 둔다.
POOL_DIR = os.environ.get('AD_POOL', '/content/ad_pool' if IN_COLAB else os.path.join(BASE_DIR, 'pool'))
FIG_DIR  = os.path.join(BASE_DIR, 'figures')
SUB_DIR  = os.path.join(BASE_DIR, 'subset')
for d in [BASE_DIR, POOL_DIR, FIG_DIR, SUB_DIR]:
    os.makedirs(d, exist_ok=True)

ANN_URL = 'https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings_annotations.json'
TAR_URL = 'https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings.tar'
ANN_PATH = os.path.join(BASE_DIR, 'amateur_drawings_annotations.json')

POOL_N    = int(os.environ.get('AD_POOL_N', 3000))   # 스트리밍으로 받을 후보 이미지 수
N_SUBSET  = int(os.environ.get('AD_N_SUBSET', 500))  # 최종 실험용 이미지 수 (300~500)
DEV_RATIO = 0.6                                       # dev 60% / test 40%
SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams['figure.dpi'] = 100
print('BASE_DIR =', BASE_DIR); print('POOL_DIR =', POOL_DIR)

## 1. 어노테이션 다운로드

In [ ]:
def download(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print('이미 있음:', path); return
    tmp = path + '.part'
    t = time.time()
    urllib.request.urlretrieve(url, tmp)
    os.rename(tmp, path)
    print(f'다운로드 완료 {os.path.getsize(path)/1e6:.0f}MB, {time.time()-t:.0f}s')

download(ANN_URL, ANN_PATH)
with open(ANN_PATH) as f:
    coco = json.load(f)
print({k: (len(v) if isinstance(v, list) else v) for k, v in coco.items()})

## 2. 데이터 구조 설명

### 2.1 파일 구성
| 파일 | 크기 | 내용 |
|---|---|---|
| `amateur_drawings_annotations.json` | 288MB | **COCO 형식** 어노테이션 (전체 178,166장) |
| `amateur_drawings.tar` | 48GB | 이미지. 이름은 `.tar`지만 실제로는 **gzip 압축 tar** (`r|gz`로 열어야 함) |

tar 내부 경로: `amateur_drawings/<uuid 첫 글자>/<uuid>.png` — JSON의 `file_name`과 동일.

### 2.2 JSON 최상위 키
| 키 | 내용 |
|---|---|
| `info` | 데이터셋 설명 (Meta FAIR, 2023) |
| `licenses` | **CC BY-NC 4.0** (비상업적 이용) — GitHub README에는 MIT라고 적혀 있으니 보고서에는 둘 다 언급 |
| `categories` | 1개: `human`. 17개 키포인트 이름 + `skeleton`(연결선, 1-based 인덱스) |
| `images` | `id`, `file_name`, `width`, `height`, `license` |
| `annotations` | `id`, `image_id`, `category_id`, `bbox`, `segmentation`, `area`, `iscrowd`, `keypoints`, `num_keypoints` |

### 2.3 annotation 필드
| 필드 | 형식 | 설명 |
|---|---|---|
| `bbox` | `[x, y, w, h]` (float) | 캐릭터 위치 박스 (좌상단 + 폭/높이) |
| `segmentation` | `[[x1,y1,x2,y2,...], ...]` | 캐릭터 영역 **폴리곤** 리스트 (대부분 1개, 일부는 여러 조각) |
| `area` | float | 마스크 면적(px²) |
| `keypoints` | 길이 51 = 17×(x, y, v) | COCO 사람 관절 순서. `v=2`(보임)만 존재 |
| `iscrowd` | 0 | 모두 0 |

이미지 1장당 어노테이션(캐릭터) 정확히 1개 → `image_id`로 1:1 매칭.

**COCO 형식이란?** 라벨을 JSON 파일에 적는 표준 양식이다. 사진 목록(`images`)과 정답 목록(`annotations`)이 **따로** 적혀 있고, `id`로 짝을 맞춘다. 라벨 17.8만 개가 파일 하나에 다 들어 있다 (사진마다 라벨 파일이 따로 있지 않다).

```json
{
  "images": [
    {"id": 0, "file_name": "amateur_drawings/c/c3772b....png", "width": 3000, "height": 4000}
  ],
  "annotations": [
    {"image_id": 0,                              // ← images의 id 0번과 짝
     "bbox": [30.1, 108.2, 290.6, 279.0],        // [x, y, 폭, 높이]
     "segmentation": [[119, 386, 109, 385, 104, 378, ...]],   // 테두리 점들 (x1,y1,x2,y2,...)
     "keypoints": [113, 204, 2, 135, 189, 2, ...]}            // 관절 17개 × (x, y, 보임여부)
  ]
}
```

### 2.4 ⚠️ 좌표계 주의 (가장 중요)
- `images[].width/height`는 **원본 사진 크기** (예: 3024×4032).
- 하지만 tar 안의 PNG는 **긴 변이 512px가 되도록 축소**되어 있다 (512보다 작은 원본은 그대로).
- **bbox · segmentation · keypoints는 모두 축소된 512px 이미지 기준 좌표**다.

| id 0번 사진 | 값 |
|---|---|
| JSON의 `width × height` | **3000 × 4000** (원본 사진 크기) |
| 실제 PNG 크기 | **384 × 512** |
| `bbox` | `[30, 108, 290, 279]` ← 512 범위 안의 숫자 = 줄어든 PNG 기준 |

- 따라서 원본 width/height로 마스크를 만들면 안 되고, **실제로 읽은 PNG 크기**로 만들어야 한다. 이 노트북이 저장하는 서브셋 JSON은 width/height를 실제 크기로 고쳐 저장하고, 원본 크기는 `orig_width/orig_height`로 남긴다.

In [ ]:
cat = coco['categories'][0]
KP_NAMES = cat['keypoints']; SKELETON = [(a-1, b-1) for a, b in cat['skeleton']]   # 0-based로 변환
print('category:', cat['name']); print('keypoints:', KP_NAMES)
print('\nimages[0] =', coco['images'][0])
a0 = dict(coco['annotations'][0])
a0['segmentation'] = f"[{len(a0['segmentation'])}개 폴리곤, 첫 폴리곤 점 {len(a0['segmentation'][0])//2}개] " + str(a0['segmentation'][0][:10]) + ' ...'
a0['keypoints'] = str(a0['keypoints'][:9]) + ' ...'
print('annotations[0] =', json.dumps(a0, indent=1, ensure_ascii=False))

## 3. 메타데이터 EDA (전체 178,166장, 이미지 없이 JSON만으로)

In [ ]:
imgs = {im['id']: im for im in coco['images']}
rows = []
for a in coco['annotations']:
    im = imgs[a['image_id']]
    W, H = im['width'], im['height']
    s = min(1.0, 512 / max(W, H))
    rw, rh = round(W * s), round(H * s)                 # 축소된 PNG 크기 추정
    x, y, w, h = a['bbox']
    kp = np.array(a['keypoints']).reshape(-1, 3)
    rows.append(dict(
        image_id=a['image_id'], file_name=im['file_name'],
        orig_w=W, orig_h=H, img_w=rw, img_h=rh,
        bx=x, by=y, bw=w, bh=h,
        area=a['area'], n_poly=len(a['segmentation']),
        n_vert=sum(len(p) // 2 for p in a['segmentation']),
        kp_in_bbox=float(np.mean((kp[:, 0] >= x - 2) & (kp[:, 0] <= x + w + 2) & (kp[:, 1] >= y - 2) & (kp[:, 1] <= y + h + 2))),
    ))
meta = pd.DataFrame(rows)
meta['aspect'] = meta.orig_w / meta.orig_h
meta['bbox_frac'] = meta.bw * meta.bh / (meta.img_w * meta.img_h)        # 박스가 이미지에서 차지하는 비율
meta['fill'] = meta.area / (meta.bw * meta.bh).clip(lower=1)             # 박스 안에서 마스크가 차지하는 비율
meta['full_bbox'] = (meta.bx <= 1) & (meta.by <= 1) & (meta.bw >= meta.img_w - 2) & (meta.bh >= meta.img_h - 2)
print(meta.describe(percentiles=[.05, .5, .95]).T[['mean', '5%', '50%', '95%']].round(3))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
ax[0,0].hist(np.log10(meta.orig_w * meta.orig_h), bins=60)
ax[0,0].set_title('Original resolution (log10 pixels)')
top = meta.aspect.round(2).value_counts().head(8)
ax[0,1].barh([str(i) for i in top.index], top.values); ax[0,1].set_title('Aspect ratio (W/H), top 8')
ax[0,2].hist(meta.bbox_frac, bins=50); ax[0,2].set_title('BBox area / image area')
ax[1,0].hist(meta.fill.clip(0, 1), bins=50); ax[1,0].set_title('Mask area / bbox area (fill)')
ax[1,1].bar(*zip(*sorted(Counter(meta.n_poly.clip(upper=6)).items()))); ax[1,1].set_title('# polygons per mask (6 = 6+)'); ax[1,1].set_yscale('log')
ax[1,2].hist(meta.n_vert.clip(upper=600), bins=50); ax[1,2].set_title('# polygon vertices (clipped 600)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/meta_eda.png', bbox_inches='tight'); plt.show()

print(f"bbox가 이미지 전체(=사용자가 박스를 수정 안 했을 가능성): {meta.full_bbox.mean()*100:.1f}%")
print(f"마스크가 여러 조각:                      {(meta.n_poly>1).mean()*100:.1f}%")
print(f"bbox가 이미지의 5% 미만(아주 작은 캐릭터): {(meta.bbox_frac<0.05).mean()*100:.1f}%")
print(f"키포인트가 bbox 밖에 있는 경우 있음:       {(meta.kp_in_bbox<1).mean()*100:.1f}%")
print(f"마스크 면적 0 (빈 폴리곤, 사실상 마스크 없음): {(meta.area<=0).sum()}개")
print(f"같은 file_name이 여러 image_id에 중복:      {meta.file_name.duplicated().sum()}개 (이미지·라벨 동일 → 고유 이미지 {meta.file_name.nunique():,}장)")
print("결측값(null/빈 값): 없음 | 관절 visibility는 모두 2(보임) → '가려진 관절' 정보는 없음")

## 4. 이미지 스트리밍 다운로드 (후보 풀)

- tar는 압축되어 있어 임의 접근이 불가능 → **앞에서부터 읽다가 `POOL_N`장 모이면 중단**.
- tar 안 순서는 uuid 폴더(c/, 1/, …) 단위인데, uuid는 랜덤이므로 앞부분 N장은 사실상 **무작위 표본**이다. (단, 같은 폴더 안에서는 `image_id`가 작은 순 → 데이터 수집 초기에 몰릴 수 있다는 한계는 보고서에 적기)
- 속도: 약 20장/초 → 3000장 ≈ 2~4분. 이미 받은 파일은 건너뛴다.

In [ ]:
by_name = {os.path.basename(im['file_name']): im['id'] for im in coco['images']}

def stream_pool(n, out_dir):
    # tar에는 어노테이션이 없는 PNG도 섞여 있음 → JSON에 있는 파일만 저장
    have = {f for f in os.listdir(out_dir) if f in by_name}
    if len(have) >= n:
        print(f'이미 {len(have)}장 있음 → 건너뜀'); return
    t = time.time(); got = len(have); no_ann = 0
    with urllib.request.urlopen(TAR_URL) as r, tarfile.open(fileobj=r, mode='r|gz') as tf:
        for m in tf:
            if not (m.isfile() and m.name.endswith('.png')): continue
            fn = os.path.basename(m.name)
            if fn not in by_name: no_ann += 1; continue
            if fn in have: continue
            with open(os.path.join(out_dir, fn), 'wb') as f:
                f.write(tf.extractfile(m).read())
            got += 1
            if got % 500 == 0: print(f'  {got}/{n}  ({time.time()-t:.0f}s)')
            if got >= n: break
    print(f'완료: {got}장, {time.time()-t:.0f}s | 어노테이션 없는 PNG {no_ann}개 건너뜀')

stream_pool(POOL_N, POOL_DIR)
pool_ids = sorted(by_name[f] for f in os.listdir(POOL_DIR) if f in by_name)
print('후보 풀 크기:', len(pool_ids))

## 5. 공통 유틸 `ad_utils.py`
팀원 모두가 **같은 방식으로** 데이터를 읽고 평가하도록 파일로 저장한다. 다른 노트북에서는:
```python
import sys; sys.path.append('/content/drive/MyDrive/cv_project/data')
from ad_utils import *
dev = load_split('/content/drive/MyDrive/cv_project/data/subset', 'dev')
```

In [ ]:
UTILS_SRC = r'''
import os, json
import numpy as np, cv2

def load_split(sub_dir, split):
    # subset/<split>.json (COCO 형식) → [(img_info, ann), ...]
    with open(os.path.join(sub_dir, f'{split}.json')) as f:
        c = json.load(f)
    anns = {a['image_id']: a for a in c['annotations']}
    return [(im, anns[im['id']]) for im in c['images']]

def load_image(path):
    # RGB uint8로 읽기 (알파 채널이 있으면 흰 배경에 합성)
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None: raise FileNotFoundError(path)
    if im.ndim == 2: im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    if im.shape[2] == 4:
        a = im[:, :, 3:4].astype(np.float32) / 255
        im = (im[:, :, :3] * a + 255 * (1 - a)).astype(np.uint8)
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

def ann_to_mask(ann, h, w):
    # COCO 폴리곤 → 0/1 마스크. 조각마다 따로 채워서 합집합 (겹침이 구멍으로 바뀌지 않게)
    m = np.zeros((h, w), np.uint8)
    for poly in ann['segmentation']:
        pts = np.round(np.array(poly, np.float32).reshape(-1, 2)).astype(np.int32)
        if len(pts) >= 3: cv2.fillPoly(m, [pts], 1)
    return m

def bbox_int(ann, h, w, pad=0):
    x, y, bw, bh = ann['bbox']
    x0, y0 = max(0, int(np.floor(x)) - pad), max(0, int(np.floor(y)) - pad)
    x1, y1 = min(w, int(np.ceil(x + bw)) + pad), min(h, int(np.ceil(y + bh)) + pad)
    return x0, y0, x1, y1

def iou(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    u = np.logical_or(p, g).sum()
    return 1.0 if u == 0 else np.logical_and(p, g).sum() / u

def _boundary(m):
    m = m.astype(np.uint8)
    return (m - cv2.erode(m, np.ones((3, 3), np.uint8))).astype(bool)

def boundary_f1(pred, gt, tol=None):
    # 경계 F1: 허용 오차 tol(px) 안에서 경계가 맞은 비율. tol 기본값 = 이미지 대각선의 0.75% (DAVIS 관례)
    if tol is None:
        tol = max(1, int(round(0.0075 * np.hypot(*gt.shape))))
    bp, bg = _boundary(pred), _boundary(gt)
    if bp.sum() == 0 and bg.sum() == 0: return 1.0
    if bp.sum() == 0 or bg.sum() == 0: return 0.0
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * tol + 1, 2 * tol + 1))
    bg_d, bp_d = cv2.dilate(bg.astype(np.uint8), k).astype(bool), cv2.dilate(bp.astype(np.uint8), k).astype(bool)
    prec = (bp & bg_d).sum() / bp.sum(); rec = (bg & bp_d).sum() / bg.sum()
    return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

def draw_overlay(img, ann, mask=None, skeleton=None, color=(255, 0, 0)):
    out = img.copy()
    if mask is None: mask = ann_to_mask(ann, *img.shape[:2])
    out[mask > 0] = (0.55 * out[mask > 0] + 0.45 * np.array(color)).astype(np.uint8)
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(out, cnts, -1, color, 1)
    x0, y0, x1, y1 = bbox_int(ann, *img.shape[:2])
    cv2.rectangle(out, (x0, y0), (x1 - 1, y1 - 1), (0, 160, 0), 2)
    kp = np.array(ann['keypoints']).reshape(-1, 3)
    if skeleton:
        for a, b in skeleton:
            cv2.line(out, tuple(map(int, kp[a, :2])), tuple(map(int, kp[b, :2])), (0, 90, 255), 2)
    for x, y, v in kp:
        if v > 0: cv2.circle(out, (int(x), int(y)), 3, (255, 220, 0), -1)
    return out
'''
with open(os.path.join(BASE_DIR, 'ad_utils.py'), 'w') as f:
    f.write(UTILS_SRC)
sys.path.insert(0, BASE_DIR)
import importlib, ad_utils; importlib.reload(ad_utils)
from ad_utils import *
print('ad_utils.py 저장:', os.path.join(BASE_DIR, 'ad_utils.py'))

In [ ]:
anns = {a['image_id']: a for a in coco['annotations']}
def pool_path(iid): return os.path.join(POOL_DIR, os.path.basename(imgs[iid]['file_name']))

# 좌표계 검증: 실제 PNG 크기 vs JSON 원본 크기, 그리고 area 필드 vs 우리가 만든 마스크 면적
chk = []
for iid in pool_ids[:200]:
    img = load_image(pool_path(iid)); h, w = img.shape[:2]
    m = ann_to_mask(anns[iid], h, w)
    chk.append(dict(json_wh=(imgs[iid]['width'], imgs[iid]['height']), png_wh=(w, h),
                    long_side=max(w, h), area_ratio=m.sum() / max(anns[iid]['area'], 1)))
chk = pd.DataFrame(chk)
print('PNG 긴 변 분포:', chk.long_side.value_counts().head().to_dict())
print('마스크 면적 / area 필드 (1에 가까워야 함): median %.3f, 5%% %.3f, 95%% %.3f' % tuple(chk.area_ratio.quantile([.5, .05, .95])))
chk.head()

In [ ]:
def show_grid(ids, ncol=6, title=None, path=None, size=2.6, label=None):
    n = len(ids); nrow = int(np.ceil(n / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(ncol * size, nrow * size * 1.25))
    for ax in np.array(axs).ravel(): ax.axis('off')
    for ax, iid in zip(np.array(axs).ravel(), ids):
        img = load_image(pool_path(iid))
        ax.imshow(draw_overlay(img, anns[iid], skeleton=SKELETON))
        ax.set_title(label(iid) if label else str(iid), fontsize=8)
    if title: fig.suptitle(title)
    plt.tight_layout()
    if path: plt.savefig(path, bbox_inches='tight')
    plt.show()

show_grid(random.sample(pool_ids, 12), title='Random samples: mask(red) / bbox(green) / skeleton', path=f'{FIG_DIR}/samples.png')

## 6. 이미지 수준 EDA — 조명 · 그림자 · 배경 복잡도
프로젝트의 핵심 문제(그림자, 불균일 조명, 복잡한 배경)를 **수치로** 잡아서 서브셋을 고르게 뽑는 데 쓴다. 배경 = GT 마스크를 넉넉히 팽창시킨 영역의 바깥.

| 특징 | 정의 | 의미 |
|---|---|---|
| `bg_bright` | 배경 밝기(L)의 중앙값 | 낮으면 어두운 사진 |
| `bg_uneven` | 배경을 4×4 격자로 나눠 각 칸 밝기 중앙값의 (최대−최소) | 크면 그림자/조명 불균일 |
| `bg_edges` | 배경에서 Canny 에지 픽셀 비율 | 크면 배경이 복잡(책상, 줄무늬 종이, 다른 물체) |
| `bg_sat` | 배경 채도 중앙값 | 크면 색 있는 종이/조명 색 틀어짐 |
| `fg_contrast` | 캐릭터 영역과 배경의 밝기 차이 | 작으면 연한 그림 → 분할 어려움 |
| `sharpness` | Laplacian 분산 | 작으면 흐린 사진 |

In [ ]:
def image_features(iid):
    img = load_image(pool_path(iid)); h, w = img.shape[:2]
    gt = ann_to_mask(anns[iid], h, w)
    k = max(3, int(0.03 * max(h, w)) | 1)
    bg = cv2.dilate(gt, np.ones((k, k), np.uint8)) == 0
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB); L = lab[:, :, 0].astype(np.float32)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f = dict(image_id=iid, h=h, w=w, fg_frac=gt.mean(), bg_frac=bg.mean())
    if bg.sum() < 0.05 * h * w:          # 배경이 거의 없으면 특징 계산 불가
        f.update(bg_bright=np.nan, bg_uneven=np.nan, bg_edges=np.nan, bg_sat=np.nan, fg_contrast=np.nan)
    else:
        cells = []
        for ys in np.array_split(np.arange(h), 4):
            for xs in np.array_split(np.arange(w), 4):
                c = bg[ys[0]:ys[-1]+1, xs[0]:xs[-1]+1]
                if c.sum() > 0.2 * c.size:
                    cells.append(np.median(L[ys[0]:ys[-1]+1, xs[0]:xs[-1]+1][c]))
        edges = cv2.Canny(cv2.GaussianBlur(gray, (3, 3), 0), 50, 150) > 0
        f.update(bg_bright=np.median(L[bg]),
                 bg_uneven=(max(cells) - min(cells)) if len(cells) >= 2 else np.nan,
                 bg_edges=edges[bg].mean(),
                 bg_sat=np.median(hsv[:, :, 1][bg]),
                 fg_contrast=np.median(L[bg]) - np.median(L[gt > 0]) if gt.sum() else np.nan)
    f['sharpness'] = cv2.Laplacian(gray, cv2.CV_64F).var()
    f['touch_border'] = bool(gt[0].any() or gt[-1].any() or gt[:, 0].any() or gt[:, -1].any())
    n_cc, _ = cv2.connectedComponents(gt)
    f['n_parts'] = n_cc - 1
    return f

feat_path = os.path.join(BASE_DIR, 'pool_features.csv')
if os.path.exists(feat_path) and len(pd.read_csv(feat_path)) == len(pool_ids):
    feat = pd.read_csv(feat_path)
else:
    t = time.time(); feat = pd.DataFrame([image_features(i) for i in pool_ids])
    feat.to_csv(feat_path, index=False); print(f'특징 계산 {time.time()-t:.0f}s')
feat = feat.merge(meta[['image_id', 'orig_w', 'orig_h', 'bbox_frac', 'fill', 'full_bbox', 'n_poly']], on='image_id')
feat.describe().T.round(3)

In [ ]:
cols = ['bg_bright', 'bg_uneven', 'bg_edges', 'bg_sat', 'fg_contrast', 'sharpness']
fig, ax = plt.subplots(1, len(cols), figsize=(18, 3))
for a, c in zip(ax, cols):
    v = feat[c].dropna(); a.hist(v.clip(upper=v.quantile(.99)), bins=40); a.set_title(c)
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/image_features.png', bbox_inches='tight'); plt.show()
print(feat[cols].corr().round(2))

### 6.1 촬영 조건 그룹 나누기
규칙 기반으로 세 그룹(+기타)으로 나눈다. 임계값은 후보 풀의 분위수로 정해서 그룹 크기가 너무 작아지지 않게 한다.
- **complex**: `bg_edges` 상위 25% — 배경이 복잡
- **shadow**: (complex 아님) `bg_uneven` 상위 30% 또는 `bg_bright` 하위 15% — 그림자/어두움/불균일
- **clean**: 나머지 중 밝고 고른 배경
- **아래 예시 그리드를 꼭 눈으로 보고** 그룹이 말이 되는지 확인. 이상하면 임계값(Q_*)을 조정.

In [ ]:
Q_EDGE, Q_UNEVEN, Q_DARK = 0.75, 0.70, 0.15
thr = dict(edge=feat.bg_edges.quantile(Q_EDGE), uneven=feat.bg_uneven.quantile(Q_UNEVEN), dark=feat.bg_bright.quantile(Q_DARK))
def condition(r):
    if pd.isna(r.bg_edges): return 'no_bg'
    if r.bg_edges >= thr['edge']: return 'complex'
    if r.bg_uneven >= thr['uneven'] or r.bg_bright <= thr['dark']: return 'shadow'
    return 'clean'
feat['condition'] = feat.apply(condition, axis=1)
print({k: round(float(v), 3) for k, v in thr.items()}); print(feat.condition.value_counts())
for c in ['clean', 'shadow', 'complex']:
    ids = feat[feat.condition == c].image_id.sample(min(6, (feat.condition == c).sum()), random_state=SEED).tolist()
    show_grid(ids, title=f'condition = {c}', path=f'{FIG_DIR}/cond_{c}.png')

## 7. 라벨 품질 점검 → 서브셋 선정 → 중복 제거 → dev/test 분할

### 7.1 제외 기준
| 플래그 | 이유 |
|---|---|
| `full_bbox` | 박스가 이미지 전체 = 사용자가 박스를 안 고쳤을 가능성 → 위치 박스로 실험 영역을 잡는 의미가 없음 |
| `no_bg` | 배경이 거의 없음(이미 크롭된 그림) → 배경 제거 실험에 부적합 |
| `fill < 0.05` | 박스 대비 마스크가 극히 작음 → 마스크 오류 가능성 (막대 인간 같은 얇은 그림은 fill 0.1 안팎이라 남김) |
| `n_parts > 3` | 마스크가 여러 조각으로 쪼개짐 → 라벨 오류 가능성 (사람형 캐릭터 1개 가정) |
| bbox 5% 미만 | 캐릭터가 너무 작음 |
| `empty_mask` | area=0 (점 3개짜리 빈 폴리곤) |

In [ ]:
feat['flag_full_bbox'] = feat.full_bbox
feat['flag_no_bg'] = feat.condition == 'no_bg'
feat['flag_low_fill'] = feat.fill < 0.05   # 0.15로 하면 정상적인 막대 인간까지 빠짐
feat['flag_empty_mask'] = feat.image_id.map(lambda i: anns[i]['area'] <= 0)
feat['flag_many_parts'] = feat.n_parts > 3
feat['flag_tiny'] = feat.bbox_frac < 0.05
flags = [c for c in feat.columns if c.startswith('flag_')]
feat['usable'] = ~feat[flags].any(axis=1)
print(feat[flags].mean().mul(100).round(1).rename('% of pool')); print('사용 가능:', feat.usable.sum(), '/', len(feat))
bad = feat[~feat.usable].sample(min(6, (~feat.usable).sum()), random_state=SEED)
show_grid(bad.image_id.tolist(), title='Examples excluded by quality flags', path=f'{FIG_DIR}/excluded.png',
          label=lambda i: ','.join(f[5:] for f in flags if feat.set_index('image_id').loc[i, f]))

### 7.2 중복/유사 그림 검사
같은 그림을 여러 번 찍어 올린 경우가 있을 수 있다 → **dHash(64bit)** 로 해밍 거리 ≤ `DUP_TH` 인 쌍을 같은 그룹으로 묶고, 그룹 단위로 dev/test를 나눈다 (한 그룹은 한쪽에만).
캐릭터 박스 영역만 해시해서 배경이 달라도 같은 그림을 잡도록 한다.
단색에 가까운 크롭은 해시가 전부 0이 되어 오탐이 나므로 제외하고, 후보 쌍은 16×16 축소 이미지 상관계수(>0.9)로 한 번 더 확인한다.

In [ ]:
def crop_gray(iid):
    img = load_image(pool_path(iid)); h, w = img.shape[:2]
    x0, y0, x1, y1 = bbox_int(anns[iid], h, w)
    return cv2.cvtColor(img[y0:y1, x0:x1], cv2.COLOR_RGB2GRAY)

def dhash(g, size=8):
    g = cv2.resize(g, (size + 1, size), interpolation=cv2.INTER_AREA)
    return (g[:, 1:] > g[:, :-1]).ravel()

def thumb(g, size=16):
    t = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    return (t - t.mean()) / (t.std() + 1e-6)

usable_ids = feat[feat.usable].image_id.tolist()
crops = [crop_gray(i) for i in usable_ids]
bits = np.stack([dhash(g) for g in crops]).astype(np.int16)
thumbs = np.stack([thumb(g) for g in crops])
dist = (bits[:, None, :] != bits[None, :, :]).sum(-1)
DUP_TH = 6
# 거의 단색인 크롭은 해시가 전부 0/1이 되어 서로 '같다'고 나오므로 비교에서 제외
informative = (bits.sum(1) >= 10) & (bits.sum(1) <= 54)
parent = list(range(len(usable_ids)))
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
cand_pairs = np.argwhere(np.triu(dist <= DUP_TH, 1))
# 2차 확인: 16x16 축소 이미지의 상관계수가 높아야 진짜 중복으로 인정
pairs = np.array([(a, b) for a, b in cand_pairs
                  if informative[a] and informative[b] and float(thumbs[a] @ thumbs[b]) / thumbs.shape[1] > 0.9]).reshape(-1, 2)
for a, b in pairs: parent[find(a)] = find(b)
feat['dup_group'] = feat.image_id.map({iid: int(find(k)) for k, iid in enumerate(usable_ids)})
print(f'해시 후보 쌍 {len(cand_pairs)}개 → 확인된 유사 쌍 {len(pairs)}개 | 2장 이상 그룹 수:', (feat.dup_group.value_counts() > 1).sum())
if len(pairs):
    sel = pairs[:4]
    show_grid([usable_ids[i] for p in sel for i in p], ncol=8, title='Near-duplicate pairs (check!)', path=f'{FIG_DIR}/duplicates.png')

### 7.3 조건별 균등 추출 + dev/test 분할
- clean / shadow / complex에서 **같은 수**씩 뽑는다 (N_SUBSET/3).
- 분할은 **중복 그룹 단위**로, 조건별 비율을 유지(층화)해서 dev 60% / test 40%.
- dev: 파라미터 튜닝·방법 개발용 / test: **4주차 최종 평가 때만** 사용 (중간에 보면서 튜닝 금지).

In [ ]:
rng = np.random.default_rng(SEED)
conds = ['clean', 'shadow', 'complex']
per = N_SUBSET // len(conds)
cand = feat[feat.usable & feat.condition.isin(conds)]
# 유사 그룹당 1장만 남김 (조건이 달라도) → 서브셋 안에 비슷한 그림이 두 장 들어가지 않음
cand = cand.sample(frac=1, random_state=SEED).drop_duplicates('dup_group')
picked = []
for c in conds:
    g = cand[cand.condition == c]
    picked.append(g.sample(min(per, len(g)), random_state=SEED))
    if len(g) < per: print(f'⚠️ {c}: {len(g)}장밖에 없음 → POOL_N을 늘리세요')
sub = pd.concat(picked).reset_index(drop=True)

sub['split'] = 'test'
for c in conds:
    groups = sub[sub.condition == c].dup_group.unique(); rng.shuffle(groups)
    n_dev = int(round(len(groups) * DEV_RATIO))
    sub.loc[sub.dup_group.isin(groups[:n_dev]) & (sub.condition == c), 'split'] = 'dev'
assert not (set(sub[sub.split == 'dev'].dup_group) & set(sub[sub.split == 'test'].dup_group)), 'dev/test 그룹 겹침!'

# 서브셋 전체를 다시 한번: dev-test 사이 해밍 거리 최솟값 확인
idx = {iid: k for k, iid in enumerate(usable_ids)}
dv = [idx[i] for i in sub[sub.split == 'dev'].image_id]; ts = [idx[i] for i in sub[sub.split == 'test'].image_id]
print('dev-test 최소 해밍 거리:', dist[np.ix_(dv, ts)].min(), f'(> {DUP_TH} 이어야 함)')
print(pd.crosstab(sub.condition, sub.split, margins=True))

In [ ]:
# 저장: 이미지 복사, GT 마스크 PNG, COCO 형식 dev/test JSON, 목록 CSV
img_out, mask_out = os.path.join(SUB_DIR, 'images'), os.path.join(SUB_DIR, 'masks_gt')
for d in [img_out, mask_out]:          # 이전 실행에서 남은 파일 제거 (설정을 바꿔 다시 돌릴 때 섞이지 않게)
    shutil.rmtree(d, ignore_errors=True)
os.makedirs(img_out, exist_ok=True); os.makedirs(mask_out, exist_ok=True)
sub_cols = ['image_id', 'split', 'condition', 'dup_group', 'h', 'w', 'orig_w', 'orig_h', 'bg_bright', 'bg_uneven', 'bg_edges', 'bg_sat', 'fg_contrast', 'sharpness', 'fill', 'bbox_frac', 'n_parts']
for split in ['dev', 'test']:
    s = sub[sub.split == split]
    out = dict(info=coco['info'], licenses=coco['licenses'], categories=coco['categories'], images=[], annotations=[])
    for r in s.itertuples():
        im = imgs[r.image_id]; fn = os.path.basename(im['file_name'])
        dst = os.path.join(img_out, fn)
        if not os.path.exists(dst): shutil.copy(pool_path(r.image_id), dst)
        mfn = fn.replace('.png', '_mask.png')
        cv2.imwrite(os.path.join(mask_out, mfn), ann_to_mask(anns[r.image_id], r.h, r.w) * 255)
        out['images'].append(dict(id=r.image_id, file_name=f'images/{fn}', mask_file=f'masks_gt/{mfn}',
                                  width=int(r.w), height=int(r.h), orig_width=im['width'], orig_height=im['height'],
                                  condition=r.condition, license=im['license']))
        out['annotations'].append(anns[r.image_id])
    with open(os.path.join(SUB_DIR, f'{split}.json'), 'w') as f: json.dump(out, f)
sub['file_name'] = sub.image_id.map(lambda i: 'images/' + os.path.basename(imgs[i]['file_name']))
sub[['file_name'] + sub_cols].to_csv(os.path.join(SUB_DIR, 'subset_list.csv'), index=False)
with open(os.path.join(SUB_DIR, 'README.txt'), 'w') as f:
    f.write(f'Amateur Drawings subset. seed={SEED}, pool={len(pool_ids)}, n={len(sub)}\n'
            'dev.json / test.json: COCO format; width/height = actual PNG size (coords are in this space)\n'
            'images/: PNG (long side <=512) | masks_gt/: GT mask (0/255) | subset_list.csv: per-image features & condition\n'
            'test split is for FINAL evaluation only (week 4).\n')
print('저장 완료:', SUB_DIR, '|', len(os.listdir(img_out)), 'images')

## 8. 평가용(test) 마스크 수동 검토
계획서: *"평가용 마스크는 직접 검토하고 오류가 있으면 수정한다."*
1. 아래 셀을 돌려 test 이미지를 페이지별로 본다 (빨강 = GT 마스크).
2. 문제 있는 id를 `REVIEW`에 적는다: `'bad'`(제외) / `'fix'`(수정 필요: 손발 누락, 배경 포함 등).
3. `'fix'` 이미지는 `masks_gt/*_mask.png`를 그림판/GIMP/CVAT 등으로 고친 뒤 `masks_gt_fixed/`에 같은 이름으로 저장 → 평가 코드는 fixed가 있으면 그것을 우선 사용.

In [ ]:
test_ids = sub[sub.split == 'test'].image_id.tolist()
PAGE = 0        # 0, 1, 2, ... 바꿔가며 확인 (페이지당 30장)
show_grid(test_ids[PAGE * 30:(PAGE + 1) * 30], ncol=6, size=2.4, title=f'test review page {PAGE}')

In [ ]:
REVIEW = {
    # 123: 'bad',   # 예: 마스크가 배경까지 포함
    # 456: 'fix',   # 예: 왼팔 누락
}
rev = pd.DataFrame({'image_id': test_ids})
rev['status'] = [REVIEW.get(i, 'ok') for i in rev.image_id]
rev.to_csv(os.path.join(SUB_DIR, 'test_review.csv'), index=False)
os.makedirs(os.path.join(SUB_DIR, 'masks_gt_fixed'), exist_ok=True)
print(rev.status.value_counts())

## 9. Sanity check: 가장 단순한 분할(Otsu)로 IoU 계산
1주차 완료 기준 *"이미지·라벨을 읽고 기본 IoU를 계산할 수 있음"* 확인용. (실제 기본 방법 A는 이미지 처리 담당이 구현 — 여기서는 파이프라인이 끝까지 도는지만 본다.)
bbox 안에서 Otsu 이진화 → 가장 큰 연결 요소 → 구멍 채우기.

In [ ]:
def otsu_baseline(img, ann):
    h, w = img.shape[:2]; x0, y0, x1, y1 = bbox_int(ann, h, w)
    g = cv2.cvtColor(img[y0:y1, x0:x1], cv2.COLOR_RGB2GRAY)
    _, b = cv2.threshold(g, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    n, lab, st, _ = cv2.connectedComponentsWithStats(b)
    if n > 1: b = (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    cnts, _ = cv2.findContours(b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    filled = np.zeros_like(b); cv2.drawContours(filled, cnts, -1, 1, -1)
    m = np.zeros((h, w), np.uint8); m[y0:y1, x0:x1] = filled
    return m

res = []
for im, ann in load_split(SUB_DIR, 'dev'):
    img = load_image(os.path.join(SUB_DIR, im['file_name']))
    gt = cv2.imread(os.path.join(SUB_DIR, im['mask_file']), 0) > 127
    t = time.time(); pred = otsu_baseline(img, ann); dt = time.time() - t
    res.append(dict(image_id=im['id'], condition=im['condition'], iou=iou(pred, gt), bf1=boundary_f1(pred, gt), ms=dt * 1000))
res = pd.DataFrame(res)
print(res.groupby('condition')[['iou', 'bf1', 'ms']].agg(['mean', 'median']).round(3))
print('\n전체 dev mean IoU = %.3f, mean BF1 = %.3f' % (res.iou.mean(), res.bf1.mean()))

In [ ]:
worst = res.nsmallest(6, 'iou')
fig, axs = plt.subplots(2, 6, figsize=(16, 6))
for k, r in enumerate(worst.itertuples()):
    img = load_image(pool_path(r.image_id)); a = anns[r.image_id]
    axs[0, k].imshow(draw_overlay(img, a)); axs[0, k].set_title(f'GT  {r.condition}', fontsize=8)
    axs[1, k].imshow(draw_overlay(img, a, mask=otsu_baseline(img, a), color=(0, 0, 255))); axs[1, k].set_title(f'Otsu IoU={r.iou:.2f}', fontsize=8)
for ax in axs.ravel(): ax.axis('off')
plt.suptitle('Worst Otsu cases on dev (failure modes preview)'); plt.tight_layout(); plt.savefig(f'{FIG_DIR}/otsu_worst.png', bbox_inches='tight'); plt.show()

## 10. 팀 공유용 요약
아래 셀 출력을 그대로 팀 채팅/보고서 초안에 붙이면 된다. 그림은 `figures/`에 저장되어 있음.

In [ ]:
print(f'''[데이터 요약]
- 전체: {len(coco["images"]):,}장, 이미지당 캐릭터 1개 (bbox + 폴리곤 마스크 + COCO 17 관절)
- 이미지: 긴 변 512px로 축소된 PNG, 좌표도 512px 기준 (JSON width/height는 원본 크기라 주의)
- 원본 해상도 중앙값 {int(meta.orig_w.median())}x{int(meta.orig_h.median())}, 다중 조각 마스크 {(meta.n_poly>1).mean()*100:.1f}%, 전체 크기 bbox {meta.full_bbox.mean()*100:.1f}%
- 후보 풀 {len(pool_ids)}장 → 품질 필터 통과 {int(feat.usable.sum())}장 → 서브셋 {len(sub)}장 (dev {int((sub.split=="dev").sum())} / test {int((sub.split=="test").sum())})
- 조건별: {sub.condition.value_counts().to_dict()}
- Otsu sanity baseline (dev): mean IoU {res.iou.mean():.3f}, BF1 {res.bf1.mean():.3f}, {res.ms.mean():.1f} ms/img
- 공유 파일: {SUB_DIR}/(dev.json, test.json, images/, masks_gt/, subset_list.csv), {BASE_DIR}/ad_utils.py
''')